# wilro-sync on Google Colab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/wiltechs-hd/wilro-sync/blob/main/notebooks/wilro_sync_colab.ipynb)

Plain-code walkthrough for [wilro-sync](https://github.com/wiltechs-hd/wilro-sync): set up → sanity checks →
prepare clips → train → lip-sync. Edit the configuration cell, then run the cells in order. Everything uses the
Python API, so you can change any setting in code (e.g. `cfg.lr_new = 5e-5`) before running a step.

| Runtime | What works |
|---|---|
| A100 40/80 GB | everything |
| L4 24 GB | training (LoRA, 512×512×49, 8-bit Adam), data prep, inference |
| T4 16 GB | tests, data prep, inference (fp16); not training |

Model downloads (~30 GB) go to local disk; clips, checkpoints and outputs go to `WORK_DIR` on Google Drive.
Data comes in three stages (`STAGE`): **smoke** (40 HDTF clips, checks everything works) → **first_model**
(HDTF + ~130 h TalkVid + MEAD pairs, ~370 GB) → **paper_scale** (~420 h, ~1 TB; use a cloud VM).
See [docs/DATASETS.md](https://github.com/wiltechs-hd/wilro-sync/blob/main/docs/DATASETS.md).

## 1. Configuration

In [ ]:
# Where the code comes from: "github" (default), "drive" (your own copy) or "local" (already at REPO)
CODE_SOURCE = "github"
GITHUB_URL = "https://github.com/wiltechs-hd/wilro-sync.git"
GIT_REF = "main"  # branch, tag or commit; pin a commit to reproduce a run
DRIVE_CODE_DIR = "/content/drive/MyDrive/wilro-sync"
REPO = "/content/wilro-sync"

# Persistent working folder (raw videos, prepared clips, checkpoints, outputs)
WORK_DIR = "/content/drive/MyDrive/wilro-sync-work"

# Data stage (configs/data/<stage>.yaml): "smoke" -> "first_model" -> "paper_scale"
STAGE = "smoke"
STAGE_OVERRIDES = []  # any stage field, e.g. ["hdtf.max_segments=80", "talkvid.hours=50"]
YOUTUBE_COOKIES = None  # cookies.txt (Netscape format) if YouTube asks to "confirm you're not a bot"
ACQUIRE_LIMIT = None  # download at most N new segments per run (None = all); re-run to continue

# Training: on top of configs/train/stage_a.yaml + the stage's train_overrides (any TrainConfig field)
TRAIN_OVERRIDES = {
    "train_mode": "lora",  # lora | adapter | full
    "lora_rank": 64,
    "grad_accum": 8,
}
RESUME = True

# Inference defaults (any InferenceConfig field, see configs/infer/default.yaml)
INFER_OVERRIDES = {"steps": 50, "tau": 0.92, "max_side": 832, "seed": 0}
DSCFG_OVERRIDES = {"omega_peak": 4.5, "gamma": 1.5}

## 2. Environment

In [ ]:
import glob
import json
import os
import shutil
import subprocess
import sys

os.environ.setdefault("HF_HOME", "/content/hf_cache")  # large model files stay on local disk
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

if "google.colab" in sys.modules and (CODE_SOURCE == "drive" or WORK_DIR.startswith("/content/drive")):
    from google.colab import drive

    drive.mount("/content/drive")
try:  # optional: a Colab secret named HF_TOKEN gives faster, rate-limit-free downloads
    from google.colab import userdata

    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    pass

RAW = os.path.join(WORK_DIR, "raw")
RUN_DIR = os.path.join(WORK_DIR, "runs", STAGE)
for d in (RAW, RUN_DIR, os.path.join(WORK_DIR, "outputs"), os.path.join(WORK_DIR, "inputs")):
    os.makedirs(d, exist_ok=True)


def sh(cmd: str, check: bool = True) -> int:
    """Run a shell command, streaming its output."""
    print(f"$ {cmd}")
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="")
    if p.wait() and check:
        raise RuntimeError(f"failed ({p.returncode}): {cmd}")
    return p.returncode


def latest_checkpoint(run_dir: str):
    steps = [d for d in sorted(glob.glob(os.path.join(run_dir, "step_*")))
             if os.path.isfile(os.path.join(d, "step.json"))]
    return steps[-1] if steps else None


import torch

if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    GPU, GPU_MEM, BF16 = props.name, props.total_memory / 2**30, torch.cuda.is_bf16_supported()
else:
    GPU, GPU_MEM, BF16 = "none", 0.0, False
CAN_TRAIN = BF16 and GPU_MEM >= 22
print(f"GPU {GPU} | {GPU_MEM:.0f} GB | bf16 {BF16} | can train: {CAN_TRAIN}")

## 3. Code and dependencies

In [ ]:
if CODE_SOURCE == "github":
    if os.path.isdir(os.path.join(REPO, ".git")):  # re-run: update in place
        sh(f"git -C {REPO} fetch -q --depth 1 origin {GIT_REF} && git -C {REPO} checkout -q FETCH_HEAD")
    else:
        shutil.rmtree(REPO, ignore_errors=True)
        sh(f"git clone -q {GITHUB_URL} {REPO} && git -C {REPO} checkout -q {GIT_REF}")
    sh(f"git -C {REPO} log -1 --format='code: %h %s (%cd)' --date=short")
elif CODE_SOURCE == "drive":
    shutil.rmtree(REPO, ignore_errors=True)
    shutil.copytree(DRIVE_CODE_DIR, REPO, ignore=shutil.ignore_patterns("__pycache__", "*.egg-info", "runs", "data"))
assert os.path.isfile(os.path.join(REPO, "pyproject.toml")), f"no repo at {REPO}"

os.chdir(REPO)
sys.path.insert(0, REPO)  # editable installs are only seen by new processes
sh(f'pip install -q -e "{".[dev,train,data]" if torch.cuda.is_available() else ".[dev,data]"}"')
import wilrosync

print("wilro-sync", wilrosync.__version__)

## 4. Sanity checks

Unit tests (CPU, ~20 s). The real-weights check downloads Wan2.1-1.3B (~6 GB) and confirms the lip-sync wrapper
reproduces it exactly before any training.

In [ ]:
sh("python -m pytest -q")

In [ ]:
sh("python scripts/check_real_wan.py")

## 5. Data stage

`configs/data/<STAGE>.yaml` decides which sources and how much of each. The steps are resumable and shared
between stages (a later stage reuses everything an earlier one downloaded, cropped, scored and prepared):

1. **plan**: segments from the official HDTF / TalkVid metadata, held-out identities, hour budget, storage estimate
2. **acquire**: download each segment and write a 512×512, 25 fps face crop to `WORK_DIR/raw/<source>/`
3. **manifest → SyncNet filter → prepare**: drop badly synced clips, fix A/V offsets, pre-compute latents

**YouTube on Colab** usually answers "confirm you're not a bot". Either set `YOUTUBE_COOKIES` to a cookies.txt
exported from your browser, or download the full videos on your own computer (see the URL-list cell) into
`WORK_DIR/raw/sources/<youtube id>.mp4`; segments are then cut locally. **MEAD** has no download link: get it from
the [project page](https://wywu.github.io/projects/MEAD/MEAD.html) and extract it to `WORK_DIR/raw/mead/`.

In [ ]:
from wilrosync.data import stages

stage = stages.load_stage(STAGE, STAGE_OVERRIDES)
plan = stages.plan_stage(stage, WORK_DIR)
print(stage.description)
print(plan["estimate"])

In [ ]:
# download + face-crop (long; safe to stop and re-run)
counts = stages.acquire(plan, WORK_DIR, cookies=YOUTUBE_COOKIES, limit=ACQUIRE_LIMIT)

In [ ]:
# Optional: list the YouTube videos still missing, to download on your own computer with
#   yt-dlp -a urls.txt -f "bv*[height<=1080]+ba/b" --merge-output-format mp4 -o "%(id)s.%(ext)s"
# then upload the files to WORK_DIR/raw/sources/ and re-run the acquire cell.
from wilrosync.data.download import export_url_list

urls_file = os.path.join(WORK_DIR, f"urls_{STAGE}.txt")
print(export_url_list(plan["train"] + plan["holdout"], urls_file, os.path.join(RAW, "sources")), "->", urls_file)

In [ ]:
manifests = stages.write_manifests(plan, WORK_DIR)
print(manifests["counts"])
prepared = stages.sync_and_prepare(plan, WORK_DIR, manifests, device="cuda" if GPU_MEM else "cpu")
print(prepared)

In [ ]:
# SyncNet scores of everything acquired so far (shared cache across stages)
import matplotlib.pyplot as plt

report = os.path.join(RAW, "syncnet_wild.report.jsonl")
scored = [r for r in map(json.loads, open(report)) if "sync_conf" in r] if os.path.isfile(report) else []
if scored:
    fig, ax = plt.subplots(1, 2, figsize=(10, 3))
    ax[0].hist([r["sync_conf"] for r in scored], bins=30)
    ax[0].axvline(stage.sync.min_conf, color="r")
    ax[0].set_title(f"SyncNet confidence ({len(scored)} clips)")
    ax[1].hist([r["av_offset"] for r in scored], bins=range(-15, 17))
    ax[1].set_title("A/V offset (frames)")
    plt.show()

## 6. Train

Builds a `TrainConfig` from `configs/train/stage_a.yaml` + the stage's data and `train_overrides` +
`TRAIN_OVERRIDES`; inspect or change `cfg` before calling `train(cfg)`. Checkpoints go to
`WORK_DIR/runs/<STAGE>/step_*`; with `RESUME` the latest one is continued after a disconnect.

In [ ]:
from wilrosync.train.config import load_config
from wilrosync.train.trainer import train

overrides = {
    **stages.train_overrides(plan, WORK_DIR),
    "data.num_workers": 2,
    "mixed_precision": "bf16" if BF16 else "no",
    "optimizer": "adamw" if GPU_MEM >= 60 else "adamw8bit",
    **TRAIN_OVERRIDES,
}
ckpt = latest_checkpoint(RUN_DIR) if RESUME else None
if ckpt:
    overrides["resume_from"] = ckpt
cfg = load_config("configs/train/stage_a.yaml", [f"{k}={v}" for k, v in overrides.items()])
print(f"stage={STAGE} mode={cfg.train_mode} steps={cfg.max_steps} batch={cfg.batch_size}x{cfg.grad_accum} "
      f"pairs={cfg.data.tds_enabled} precision={cfg.mixed_precision} optim={cfg.optimizer} resume={cfg.resume_from}")

In [ ]:
assert CAN_TRAIN, f"{GPU} can't train this model (needs bf16 and >= 22 GB): use an L4 or A100 runtime"
result = train(cfg)
print("finished at step", result["steps"])

In [ ]:
import matplotlib.pyplot as plt

log = [json.loads(line) for line in open(os.path.join(RUN_DIR, "log.jsonl")) if line.strip()]
plt.figure(figsize=(8, 3.5))
plt.plot([r["step"] for r in log], [r["loss"] for r in log])
plt.xlabel("step")
plt.ylabel("flow-matching loss")
plt.grid(alpha=0.3)
plt.title(STAGE)
plt.show()

## 7. Lip sync

`lipsync(video, audio, bbox=None)` runs the pipeline with the latest checkpoint of this stage (EMA weights if
present). Pass `bbox=(x0, y0, x1, y1)` in source-video pixels to sync only that face in a multi-face scene;
`preview(video, bbox)` shows the first frame with a grid and the box. Until a checkpoint is properly trained, the
output is not lip-synced.

In [ ]:
from base64 import b64encode

import matplotlib.patches as patches
from IPython.display import HTML, display

from wilrosync.io.media import read_audio, read_video, write_video
from wilrosync.pipeline.lipsync import LipSyncPipeline, load_infer_config
from wilrosync.pipeline.target import TargetSignals

_pipe = {}


def load_pipeline(ckpt: str | None = None, use_ema: bool = True) -> LipSyncPipeline:
    ckpt = ckpt or latest_checkpoint(RUN_DIR)
    assert ckpt, f"no checkpoint in {RUN_DIR}"
    if use_ema and os.path.isdir(os.path.join(ckpt, "ema")):
        ckpt = os.path.join(ckpt, "ema")
    if _pipe.get("ckpt") != ckpt:
        _pipe.clear()
        dtype = torch.bfloat16 if BF16 else torch.float16
        _pipe.update(ckpt=ckpt, pipe=LipSyncPipeline.from_pretrained(ckpt, device="cuda", dtype=dtype))
    return _pipe["pipe"]


def preview(video: str, bbox=None) -> None:
    frame = read_video(video, max_frames=1)[0][0]
    h, w = frame.shape[:2]
    fig, ax = plt.subplots(figsize=(9, 9 * h / w))
    ax.imshow(frame)
    ax.set_xticks(range(0, w, max(1, w // 10)))
    ax.set_yticks(range(0, h, max(1, h // 10)))
    ax.grid(alpha=0.4)
    if bbox:
        x0, y0, x1, y1 = bbox
        ax.add_patch(patches.Rectangle((x0, y0), x1 - x0, y1 - y0, fill=False, lw=2, ec="lime"))
    plt.show()


def lipsync(video: str, audio: str, bbox=None, out: str | None = None, ckpt: str | None = None, **overrides) -> str:
    pipe = load_pipeline(ckpt)
    frames, fps = read_video(video)
    target = TargetSignals.from_bbox(len(frames), *frames.shape[1:3], bbox) if bbox else None
    icfg = load_infer_config("configs/infer/default.yaml")
    for k, v in {**INFER_OVERRIDES, **overrides}.items():
        setattr(icfg, k, v)
    for k, v in DSCFG_OVERRIDES.items():
        setattr(icfg.dscfg, k, v)
    result = pipe(frames, fps, read_audio(audio), target=target, cfg=icfg)
    out = out or os.path.join(WORK_DIR, "outputs", os.path.splitext(os.path.basename(video))[0] + "_synced.mp4")
    write_video(out, result, fps, audio_path=audio)
    print(f"{len(result)} frames -> {out}")
    return out


def show(path: str, width: int = 720) -> None:
    data = b64encode(open(path, "rb").read()).decode()
    display(HTML(f'<video width="{width}" controls src="data:video/mp4;base64,{data}"></video>'))

In [ ]:
# Example: put your files in WORK_DIR/inputs/ and edit the paths / box below.
video = os.path.join(WORK_DIR, "inputs", "two_people.mp4")
audio = os.path.join(WORK_DIR, "inputs", "speech.wav")
bbox = None  # e.g. (410, 80, 620, 330) for the target face; None = whole frame (single speaker)

preview(video, bbox)
out = lipsync(video, audio, bbox=bbox)
show(out)

In [ ]:
# Lip-sync metrics (SyncNet): LSE-C higher is better, LSE-D lower is better; bbox -> target face only
from wilrosync.eval.syncnet import SyncNet, lse

scorer = SyncNet()
print("source :", lse(video, audio, bbox=bbox, scorer=scorer))  # original lips vs the new audio
print("output :", lse(out, bbox=bbox, scorer=scorer))

### Held-out evaluation

Re-dubs each held-out clip (identities never seen in training) with the audio of the next one and compares
LSE-C / LSE-D of the untouched source (lips don't match the new audio), the output, and the source with its
own audio (an upper bound).

In [ ]:
import numpy as np


def evaluate_holdout(n: int = 8, steps: int = 30) -> dict:
    rows = [json.loads(line) for line in open(os.path.join(WORK_DIR, "stages", STAGE, "holdout.jsonl"))][: n + 1]
    res = {"mismatched source": [], "output": [], "matched source": []}
    for a, b in zip(rows, rows[1:]):
        video, speech = os.path.join(RAW, a["video"]), os.path.join(RAW, b["video"])
        out = lipsync(video, speech, steps=steps,
                      out=os.path.join(WORK_DIR, "outputs", "holdout", f"{a['id']}__{b['id']}.mp4"))
        res["mismatched source"].append(lse(video, speech, scorer=scorer, assume_cropped=True))
        res["output"].append(lse(out, scorer=scorer, assume_cropped=True))
        res["matched source"].append(lse(video, scorer=scorer, assume_cropped=True))
    summary = {k: {m: round(float(np.nanmean([r[m] for r in v])), 3) for m in ("lse_c", "lse_d")}
               for k, v in res.items() if v}
    for k, v in summary.items():
        print(f"{k:18s} LSE-C {v['lse_c']:6.3f}   LSE-D {v['lse_d']:6.3f}")
    return summary


holdout_scores = evaluate_holdout(n=8)

## Notes

* After a disconnect: run sections 1–3, then continue where you were (acquire, prepare and training all resume).
* Moving to the next stage: set `STAGE = "first_model"` and run section 5 again; already processed data is reused.
* Out of memory while training: `STAGE_OVERRIDES = ["clips.size=384"]` (re-prepares), or `train_mode="adapter"`.
* Out of memory at inference: `lipsync(..., max_side=640)`.
* Only process footage you have the rights and consent to edit, and label results as synthetic.